# Export Aggregated KL Metrics

This notebook reuses the same aggregation logic as `plot_rank_kl_summary.py` and organizes one sheet/tab per model.

It supports three outputs:
- preview tables inside Jupyter
- one CSV per model tab
- an Excel-compatible workbook
- optional Google Sheets upload if `gspread` and Google auth are available


In [1]:
from __future__ import annotations

import csv
import sys
from pathlib import Path
from xml.sax.saxutils import escape

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'state_spectrum_sweep').exists():
    raise RuntimeError('Start JupyterLab from the repo root so state_spectrum_sweep is importable.')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from state_spectrum_sweep.plot_rank_kl_summary import (
    EXPERIMENTS_ROOT,
    PERCENTILES,
    collect_run_stats_for_directories,
)

try:
    import pandas as pd
except ImportError:
    pd = None


## Configure runs

Set `EXPERIMENTS_DIR` to your remote experiments folder and add one entry per model tab.


In [2]:
EXPERIMENTS_DIR = EXPERIMENTS_ROOT

MODEL_RUN_GROUPS = {
    'mamba2-2.7b': [
        'mamba2_hf_rank4',
        'mamba2_hf_rank8',
        'mamba2_ssm_rank16',
        'mamba2_ssm_rank32',
        'mamba2_ssm_rank64',
    ],
    'Qwen3.5-4B': [
        'suite_kl_20260419_053402',
        'suite_kl_20260419_083118',
        'suite_kl_20260419_113650',
        'suite_kl_20260419_150958',
        'suite_kl_20260419_195735',
    ],
    'Qwen3.6-27B': [
        'suite_qwen36_27b_rank16_hf_full_20260425_055548',
        'suite_qwen36_27b_rank32_hf_full_20260424_155345',
        'suite_qwen36_27b_rank4_hf_full_20260426_011403',
        'suite_qwen36_27b_rank64_hf_full_20260429_013437',
        'suite_qwen36_27b_rank8_hf_full_20260425_161343',
    ],
    # 'Zamba': [
    #     'zamba_rank4_run_name',
    #     'zamba_rank8_run_name',
    # ],
}

OUTPUT_DIR = REPO_ROOT / 'state_spectrum_sweep' / 'exports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXCEL_XML_PATH = OUTPUT_DIR / 'kl_rank_summary.xml'

CSV_DIR = OUTPUT_DIR / 'csv'
CSV_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
def resolve_run_dir(experiments_root: Path, run_name: str) -> Path:
    candidate = Path(run_name).expanduser()
    if candidate.is_absolute():
        return candidate.resolve()
    return (experiments_root / run_name).resolve()


def sanitize_worksheet_name(name: str) -> str:
    invalid = set('[]:*?/\\')
    sanitized = ''.join('_' if char in invalid else char for char in name).strip()
    return (sanitized or 'Sheet')[:31]


def build_rows(model_label: str, runs):
    rows = []
    for run in runs:
        row = {
            'model': model_label,
            'rank': run.x_value,
            'run_dir': run.run_dir.name,
            'prompt_count': run.prompt_count,
        }
        for label in PERCENTILES:
            row[label] = run.values[label]
        rows.append(row)
    return rows


def collect_tables(experiments_root: Path, groups: dict[str, list[str]]):
    tables = {}
    for model_label, run_names in groups.items():
        run_dirs = [resolve_run_dir(experiments_root, run_name) for run_name in run_names]
        runs = collect_run_stats_for_directories(run_dirs)
        rows = build_rows(model_label, runs)
        if pd is not None:
            tables[model_label] = pd.DataFrame(rows)
        else:
            tables[model_label] = rows
    return tables


def write_csv_tables(tables, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    for model_label, table in tables.items():
        path = output_dir / f'{sanitize_worksheet_name(model_label)}.csv'
        if pd is not None and hasattr(table, 'to_csv'):
            table.to_csv(path, index=False)
        else:
            rows = table
            if not rows:
                continue
            with path.open('w', encoding='utf-8', newline='') as handle:
                writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
                writer.writeheader()
                writer.writerows(rows)
        print(f'Wrote {path}')


def xml_cell(value):
    if value is None:
        return '<Cell/>'
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return f'<Cell><Data ss:Type="Number">{value}</Data></Cell>'
    return f'<Cell><Data ss:Type="String">{escape(str(value))}</Data></Cell>'


def write_excel_xml(tables, output_path: Path):
    workbook = (
        '<?xml version="1.0"?>'
        '<?mso-application progid="Excel.Sheet"?>'
        '<Workbook xmlns="urn:schemas-microsoft-com:office:spreadsheet" '
        'xmlns:o="urn:schemas-microsoft-com:office:office" '
        'xmlns:x="urn:schemas-microsoft-com:office:excel" '
        'xmlns:ss="urn:schemas-microsoft-com:office:spreadsheet" '
        'xmlns:html="http://www.w3.org/TR/REC-html40">'
    )
    worksheets = []
    for model_label, table in tables.items():
        if pd is not None and hasattr(table, 'columns'):
            rows = [list(table.columns)] + table.values.tolist()
        else:
            if not table:
                continue
            rows = [list(table[0].keys())] + [list(row.values()) for row in table]
        row_xml = ['<Row>' + ''.join(xml_cell(value) for value in row) + '</Row>' for row in rows]
        worksheet = (
            f'<Worksheet ss:Name="{escape(sanitize_worksheet_name(model_label))}">'
            '<Table>' + ''.join(row_xml) + '</Table></Worksheet>'
        )
        worksheets.append(worksheet)
    workbook += ''.join(worksheets) + '</Workbook>'
    output_path.write_text(workbook, encoding='utf-8')
    print(f'Wrote {output_path}')


def upload_to_google_sheets(tables, spreadsheet_id: str, credentials_path: str | Path):
    import gspread
    from google.oauth2.service_account import Credentials

    scopes = ['https://www.googleapis.com/auth/spreadsheets']
    credentials = Credentials.from_service_account_file(str(credentials_path), scopes=scopes)
    client = gspread.authorize(credentials)
    spreadsheet = client.open_by_key(spreadsheet_id)

    for model_label, table in tables.items():
        worksheet_name = sanitize_worksheet_name(model_label)
        if pd is not None and hasattr(table, 'columns'):
            values = [list(table.columns)] + table.fillna('').values.tolist()
        else:
            if not table:
                values = [['no rows found']]
            else:
                values = [list(table[0].keys())] + [list(row.values()) for row in table]

        try:
            worksheet = spreadsheet.worksheet(worksheet_name)
            worksheet.clear()
        except gspread.WorksheetNotFound:
            worksheet = spreadsheet.add_worksheet(
                title=worksheet_name,
                rows=max(len(values) + 20, 100),
                cols=max(len(values[0]) + 5, 20),
            )

        worksheet.update(values, 'A1')
        print(f'Uploaded worksheet {worksheet_name}')


## Build the tables

Run this cell to aggregate KL summaries across ranks for each model.


In [4]:
tables = collect_tables(EXPERIMENTS_DIR, MODEL_RUN_GROUPS)
list(tables.keys())


SystemExit: Run directory does not exist: /home/yifeif/sssm-states/state_spectrum_sweep/experiments/mamba2_ssm_rank64

/home/yifeif/sssm-states/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
for model_label, table in tables.items():
    print(f'\n=== {model_label} ===')
    display(table if pd is not None else table)


## Export to CSV and Excel

These exports do not require Google auth.


In [ ]:
write_csv_tables(tables, CSV_DIR)
write_excel_xml(tables, EXCEL_XML_PATH)


## Optional Google Sheets upload

This still needs Google Sheets credentials. Running in JupyterLab helps with inspection and iteration, but it does not remove the auth requirement.

If you have a service account JSON key and `gspread` plus `google-auth` available in that environment, fill these in and run the cell.


In [ ]:
GOOGLE_SHEET_ID = ''
GOOGLE_CREDENTIALS_JSON = ''

# upload_to_google_sheets(
#     tables=tables,
#     spreadsheet_id=GOOGLE_SHEET_ID,
#     credentials_path=GOOGLE_CREDENTIALS_JSON,
# )
